In [ ]:
# Imports and paths setup
import sys
import rp
import torch
import numpy as np
from einops import rearrange

top_dir = rp.get_git_toplevel()
ltx_dir = rp.path_join(top_dir, 'LTX2')
ltx_src = rp.path_join(ltx_dir, 'src')
nfs_models_dir = rp.path_join(ltx_dir, 'models')

# Add project source code to path
sys.path += [nfs_models_dir]
sys.path += rp.path_join(ltx_src, 'packages', ['ltx-core', 'ltx-trainer', 'ltx-pipelines'], 'src')

from download_models import local_download_dir, download_from_web
models_dir = local_download_dir

# LTX Pipeline imports
from ltx_core.loader import LTXV_LORA_COMFY_RENAMING_MAP, LoraPathStrengthAndSDOps
from ltx_core.model.video_vae import TilingConfig
from ltx_core.conditioning import VideoConditionByKeyframeIndex
from ltx_pipelines.ic_lora import ICLoraPipeline
from ltx_pipelines.utils.media_io import encode_video, resize_and_center_crop, normalize_latent
from ltx_pipelines.utils.constants import AUDIO_SAMPLE_RATE

# Model paths
checkpoint_path        = rp.path_join(models_dir, "ltx-2-19b-distilled.safetensors")
# checkpoint_path        = rp.path_join(models_dir, "ltx-2-19b-dev.safetensors")
# checkpoint_path        = rp.path_join(models_dir, "ltx-2-19b-dev-fp8.safetensors")
# checkpoint_path        = rp.path_join(models_dir, "ltx-2-19b-dev-fp8.safetensors")
# distill_lora_path      = rp.path_join(models_dir, "ltx-2-19b-distilled-lora-resized_dynamic_fro095_avg_rank_242_bf16.safetensors")
distill_lora_path      = rp.path_join(models_dir, "ltx-2-19b-distilled-lora-384.safetensors")
spatial_upsampler_path = rp.path_join(models_dir, "ltx-2-spatial-upscaler-x2-1.0.safetensors")
canny_lora_path        = rp.path_join(models_dir, "ltx-2-19b-ic-lora-canny-control.safetensors")
detailer_lora_path     = rp.path_join(models_dir, "ltx-2-19b-ic-lora-detailer.safetensors")
gemma_root             = models_dir

# Output directory
output_dir = rp.path_join(top_dir, "outputs")
rp.make_directory(output_dir)

In [ ]:
# Setup
IN_NOTEBOOK = rp.running_in_jupyter_notebook()
DEVICE = rp.select_torch_device(prefer_used=True, reserve=True)
DTYPE = torch.bfloat16
download_from_web()
rp.r._ensure_ffmpeg_installed()

In [ ]:
# Helpers
def show_video(video):
    if IN_NOTEBOOK:
        rp.display_video(video)

def save_video_with_audio(video_tensor, audio_tensor, path, fps=25):
    encode_video(
        video=video_tensor,
        fps=int(fps),
        audio=audio_tensor,
        audio_sample_rate=AUDIO_SAMPLE_RATE,
        output_path=path,
        video_chunks_number=1,
    )

def frames_to_conditioning_tensor(frames, height, width, dtype=torch.bfloat16, device=None):
    """
    Convert a list of numpy frames (H W C uint8) to the tensor format
    expected by the video encoder for IC-LoRA conditioning.
    Returns tensor of shape (1, C, T, H, W) in [-1, 1] range.
    """
    if device is None:
        device = DEVICE
    result = None
    for frame in frames:
        t = torch.tensor(frame, dtype=torch.float32, device=device)
        t = resize_and_center_crop(t, height, width)
        t = normalize_latent(t, device, dtype)
        result = t if result is None else torch.cat([result, t], dim=2)
    return result

In [ ]:
# Create Pipeline (run once)
# IC-LoRA: Canny edge control on Stage 1, no LoRA on Stage 2 (upscale only)

canny_lora    = LoraPathStrengthAndSDOps(   canny_lora_path, 1, LTXV_LORA_COMFY_RENAMING_MAP)
detailer_lora = LoraPathStrengthAndSDOps(detailer_lora_path, 1, LTXV_LORA_COMFY_RENAMING_MAP)
# distill_lora  = LoraPathStrengthAndSDOps( distill_lora_path, 1, LTXV_LORA_COMFY_RENAMING_MAP)

pipeline = ICLoraPipeline(
    checkpoint_path=checkpoint_path,
    spatial_upsampler_path=spatial_upsampler_path,
    gemma_root=gemma_root,
    loras=[canny_lora, detailer_lora],  # Canny IC-LoRA on Stage 1
)

# Patch _create_conditionings to accept numpy frames directly (not just file paths)
# This avoids lossy mp4 round-trip which smears canny edges
from ltx_pipelines.utils.helpers import image_conditionings_by_replacing_latent

_original_create_conditionings = pipeline._create_conditionings

def _create_conditionings_tensor(self, images, video_conditioning, height, width, num_frames, video_encoder):
    conditionings = image_conditionings_by_replacing_latent(
        images=images, height=height, width=width,
        video_encoder=video_encoder, dtype=self.dtype, device=self.device,
    )
    for frames_or_path, strength in video_conditioning:
        if isinstance(frames_or_path, str):
            from ltx_pipelines.utils.media_io import load_video_conditioning
            video = load_video_conditioning(
                video_path=frames_or_path, height=height, width=width,
                frame_cap=num_frames, dtype=self.dtype, device=self.device,
            )
        else:
            # frames_or_path is a list of numpy frames (H W C uint8)
            video = frames_to_conditioning_tensor(frames_or_path[:num_frames], height, width, self.dtype, self.device)
        encoded_video = video_encoder(video)
        conditionings.append(VideoConditionByKeyframeIndex(keyframes=encoded_video, frame_idx=0, strength=strength))
    return conditionings

import types
pipeline._create_conditionings = types.MethodType(_create_conditionings_tensor, pipeline)

In [ ]:
# Load source video and create canny edges
# Uses the same pexels boat video as VAE_Test notebook
# We keep everything as numpy/tensors — no lossy mp4 round-trip

video_path = rp.download_to_cache('https://www.pexels.com/download/video/5291434/')
source_frames = rp.load_video(video_path, length=150, show_progress=True)

# Resize and crop to VAE requirements (same as VAE_Test)
source_frames = rp.resize_images_to_fit(source_frames, height=512, width=768, allow_growth=False)
source_frames = rp.as_numpy_array(source_frames)
H, W = source_frames.shape[1:3]
source_frames = rp.crop_images(source_frames, (H // 32) * 32, (W // 32) * 32, origin='center')
source_frames = rp.as_numpy_array(source_frames)
T = len(source_frames)
source_frames = source_frames[:1 + 8 * ((T - 1) // 8)]
print(f"Source video: {source_frames.shape}")

# Compute canny edges
canny_frames = [rp.as_rgb_image(rp.auto_canny(frame)) for frame in rp.eta(source_frames, 'Canny')]
print(f"Canny frames: {len(canny_frames)}, shape {canny_frames[0].shape}")
show_video(canny_frames)

In [ ]:
# Generate video conditioned on canny edges
# Two-stage: Stage 1 at half-res with canny conditioning -> Upscale to full-res
# Dimensions must be divisible by 64
# IMPORTANT: prompt should describe content that matches the structure of the canny edges!

prompt = "A speedboat cutting through ocean waves on a bright sunny day, blue water, white foam spray, tropical coastline, cinematic"
prompt = ""

height, width, num_frames, frame_rate = 768, 1280, 121, 25.0
# height, width, num_frames, frame_rate = 1088, 1920, 121+8*3, 25.0
height, width, num_frames, frame_rate = 1472, 2560, 121+8*3, 25.0
height, width, num_frames, frame_rate = 1472, 2560, 121, 25.0

# video_conditioning: list of (frames_or_path, strength) tuples
# Pass numpy frames directly — no mp4 round-trip needed
# strength controls how strongly the canny edges influence generation (0.0 to 1.0)
canny_strength = 1.0

ltx_video, ltx_audio = pipeline(
    prompt=prompt,
    seed=42,
    height=height,
    width=width,
    num_frames=num_frames,
    frame_rate=frame_rate,
    # num_inference_steps=10,
    # cfg_guidance_scale=4.0,
    images=[],
    video_conditioning=[(canny_frames, canny_strength)],
    tiling_config=TilingConfig.default(),
)

with torch.inference_mode():
    video_tensor = torch.cat(list(ltx_video), dim=0)

video_path = rp.path_join(output_dir, "ic_lora_canny_output.mp4")
video_path = rp.get_unique_copy_path(video_path)
save_video_with_audio(video_tensor, ltx_audio, video_path, frame_rate)
video = rp.as_numpy_array(video_tensor)

In [ ]:
show_video(video_path)

In [ ]:
rp.ntfy_send('notebook inference done')